# Matched 700-sign GRU comparison

Train two separate models from scratch on the same 700 ASL Citizen glosses: 2 GRU layers Р“вЂ” 256 units and 5 GRU layers Р“вЂ” 512 units. Both use the same 254 landmark features, 23 geometry features, temporal mean pooling, temporal/spatial augmentation, dropout 0.5, AdamW and frozen linear logistic readout. Class selection includes the existing 15 signs, then prioritizes available official **training** examples (ties alphabetical); validation presence is required. No test performance influences selection. Official signer-disjoint splits are preserved.

For this larger task, select by minimum validation loss with patience 40 after at least 38 epochs and a 300-epoch cap instead of the previous tiny-split PLEASE/MY rule. The test split has been examined during previous full-vocabulary work, so scores are not a fresh independent evaluation. Model code and training loops are implemented below. A run directory is never overwritten.

Each clip is limited to 83 cached landmark frames, the rounded full-dataset average (82.76). Longer sequences are uniformly sampled with endpoints preserved; shorter ones are not upsampled. The source cache already has a 128-frame cap, so clips originally longer than 128 undergo two successive sampling steps, not direct sampling from raw video. The same 83-frame rule is applied to train, validation, readout fitting and test.


In [1]:
from pathlib import Path
import os,sys,json,time,random,hashlib,warnings,copy,collections
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader,Dataset
from torch.nn.utils.rnn import pack_padded_sequence,pad_packed_sequence
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import precision_recall_fscore_support,confusion_matrix
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/sign_language').is_dir());sys.path.insert(0,str(ROOT/'src'))
from sign_language.expanded_pilot import SIGNS
from sign_language.training import backward_step,cuda_check
from sign_language.regularization import augment_batch,PROFILES
from sign_language.temporal_augmentation import TemporalCollator,CONFIG as TEMPORAL_CONFIG
from sign_language.trajectory_geometry import trajectory_batch,fit_statistics,FEATURES
from sign_language.five_sign_pilot import sha256
PARENT=ROOT/'data/training/asl-gru-128-v1';OUTPUT=ROOT/'models/asl-700-sign-gru-comparison-v1'
manifest=json.loads((PARENT/'manifest.json').read_text());status=json.loads((PARENT/'status.json').read_text());assert status['status']=='complete' and status['cache_id']==manifest['cache_id']
indexes={s:json.loads((PARENT/f'{s}-index.json').read_text()) for s in ('train','val')}
counts={s:collections.Counter(r['gloss'] for r in indexes[s]) for s in indexes}
assert all(counts['train'][s] and counts['val'][s] for s in SIGNS)
others=sorted((s for s in counts['train'] if counts['val'][s] and s not in SIGNS),key=lambda s:(-counts['train'][s],s))
names=list(SIGNS)+others[:700-len(SIGNS)];labels={s:i for i,s in enumerate(names)}
assert len(labels)==700
assert not OUTPUT.exists(),f'{OUTPUT} already exists; choose a new run directory'
assert torch.cuda.is_available();torch.set_num_threads(2);torch.backends.cudnn.enabled=False;torch.backends.cudnn.benchmark=False
print('GPU:',cuda_check());print('Classes:',len(names),'train:',sum(counts['train'][s] for s in names),'validation:',sum(counts['val'][s] for s in names))

GPU: {'torch': '2.11.0+cu128', 'cuda_runtime': '12.8', 'gpu': 'NVIDIA GeForce RTX 4060', 'vram_bytes': 8585216000}
Classes: 700 train: 11355 validation: 2413


## Datasets and integrity checks

Read from the existing prepared landmark cache. The original videos and landmark extraction are untouched. Parent chunks used by the chosen rows are checked against their recorded hashes; labels and signer separation are verified. The test split is opened only after both models have finished training.

In [2]:
from sign_language.expanded_pilot import digest_chunk
class SelectedDataset(Dataset):
    def __init__(self,split,index):
        self.split=split;self.index=index;self.rows=[i for i,r in enumerate(index) if r['gloss'] in labels]
        self.arrays=[np.load(PARENT/f'{split}-{n}.npy',mmap_mode='r') for n in ('features','lengths','targets')]
        assert all(len(a)==len(index) for a in self.arrays)
        needed=set(self.rows)
        for marker in sorted(PARENT.glob(f'{split}-*.prepared.json')):
            record=json.loads(marker.read_text());start,count=record['start'],record['count'];covered=needed.intersection(range(start,start+count))
            if covered:
                assert record['cache_id']==manifest['cache_id'] and digest_chunk(self.arrays,start,count)==record['sha256'],marker
                needed.difference_update(covered)
        assert not needed
        assert all(int(self.arrays[2][i])==manifest['labels'][index[i]['gloss']] for i in self.rows)
    def __len__(self):return len(self.rows)
    def __getitem__(self,j):
        i=self.rows[j];return torch.from_numpy(np.array(self.arrays[0][i],dtype=np.float32)),int(self.arrays[1][i]),labels[self.index[i]['gloss']]
train_data=SelectedDataset('train',indexes['train']);val_data=SelectedDataset('val',indexes['val'])
people={s:{indexes[s][i]['participant_id'] for i in dataset.rows} for s,dataset in [('train',train_data),('val',val_data)]}
assert not people['train']&people['val']
OUTPUT.mkdir(parents=True,exist_ok=False)
(OUTPUT/'labels.json').write_text(json.dumps(labels,indent=2));(OUTPUT/'class-selection.json').write_text(json.dumps({'rule':'15 pilot signs then highest train counts with validation presence; alphabetical tie','parent_cache_id':manifest['cache_id'],'sequence_frame_limit':83,'signs':names,'counts':{s:{'train':counts['train'][s],'val':counts['val'][s]} for s in names}},indent=2))
print('Verified rows:',len(train_data),len(val_data),'signers:',{s:len(v) for s,v in people.items()})

Verified rows: 11355 2413 signers: {'train': 35, 'val': 6}


## Model definition

Each model has a unidirectional stacked GRU, masked mean pooling over real frames, standardized geometry and a linear classifier.

In [3]:
class DeepTrajectoryGRU(nn.Module):
    def __init__(self, input_size=254, hidden_size=256, layers=2, classes=700, dropout=.5):
        super().__init__()
        self.gru=nn.GRU(input_size,hidden_size,num_layers=layers,batch_first=True,dropout=dropout)
        self.dropout=nn.Dropout(dropout)
        self.classifier=nn.Linear(hidden_size+len(FEATURES),classes)
        for name in ('median','mean','scale'):
            self.register_buffer('geometry_'+name,torch.ones(len(FEATURES)) if name=='scale' else torch.zeros(len(FEATURES)))
    def set_statistics(self,stats):
        for name in ('median','mean','scale'):
            value=torch.tensor(stats[name],dtype=torch.float32,device=self.geometry_mean.device)
            assert value.shape==(len(FEATURES),) and torch.isfinite(value).all()
            if name=='scale': assert (value>0).all()
            getattr(self,'geometry_'+name).copy_(value)
    def encode(self,frames,lengths):
        packed=pack_padded_sequence(frames,lengths.cpu(),batch_first=True,enforce_sorted=False)
        packed,_=self.gru(packed)
        values,actual=pad_packed_sequence(packed,batch_first=True)
        mask=(torch.arange(values.shape[1],device=values.device)[None,:]<actual.to(values.device)[:,None]).unsqueeze(-1)
        return (values*mask).sum(1)/actual.to(values.device).clamp_min(1)[:,None]
    def standardized_geometry(self,geometry):
        return (torch.where(torch.isnan(geometry),self.geometry_median,geometry)-self.geometry_mean)/self.geometry_scale
    def forward(self,frames,lengths):
        pooled=self.dropout(self.encode(frames,lengths))
        geometry=torch.as_tensor(trajectory_batch(frames,lengths),device=frames.device,dtype=torch.float32)
        values=torch.cat((pooled,self.standardized_geometry(geometry)),dim=1)
        return self.classifier(values)

## Prepare geometry statistics

Fit median imputation and standardization only on the selected training clips; compute geometry again after each training augmentation.

In [4]:
MAX_FRAMES=83
class FrameLimitCollator:
    def __init__(self,augment=False): self.temporal=TemporalCollator(enabled=augment)
    def __call__(self,samples):
        frames,lengths,targets=self.temporal(samples)
        limited=torch.zeros((len(frames),MAX_FRAMES,frames.shape[2]),dtype=frames.dtype)
        lengths_out=[]
        for i,length in enumerate(lengths.tolist()):
            take=min(int(length),MAX_FRAMES)
            if int(length)>MAX_FRAMES:
                indices=torch.linspace(0,int(length)-1,MAX_FRAMES).round().long()
                limited[i,:take]=frames[i,indices]
            else: limited[i,:take]=frames[i,:take]
            lengths_out.append(take)
        return limited,torch.tensor(lengths_out,dtype=torch.long),targets

clean_loader=DataLoader(train_data,batch_size=16,shuffle=False,num_workers=0,collate_fn=FrameLimitCollator())
val_loader=DataLoader(val_data,batch_size=16,shuffle=False,num_workers=0,collate_fn=FrameLimitCollator())
geometry=np.concatenate([trajectory_batch(x,l) for x,l,_ in clean_loader]);statistics=fit_statistics(geometry)
print('Training geometry:',geometry.shape,'missing:',int(np.isnan(geometry).sum()))
(OUTPUT/'geometry-statistics.json').write_text(json.dumps(statistics,indent=2))

Training geometry: (11355, 23) missing: 4809


## Train both GRUs

Same train/validation samples and configuration for both models; random augmentation sequences can differ because model sizes consume random numbers differently. The loader uses zero workers for reliability in Windows notebooks. Each model keeps only the best validation-loss checkpoint and a latest-epoch recovery checkpoint during training. The second-stage readout is fitted after both backbones have trained.

Training uses 150 batches of four clips per epoch (600 clips). A seeded shuffled sampler carries its position across epochs, covering all 11,355 training clips once before reshuffling. Three hundred epochs allow about 15.85 passes; individual classes need not appear in every short epoch. All 2,413 validation clips are evaluated after each epoch. Early stopping uses patience 40 after a 38-epoch minimum.


In [ ]:
class ContinuousShuffledSampler(torch.utils.data.Sampler):
    def __init__(self, count, samples_per_epoch=600, seed=42):
        self.count=count;self.samples_per_epoch=samples_per_epoch
        self.generator=torch.Generator().manual_seed(seed)
        self.order=torch.randperm(count,generator=self.generator).tolist();self.position=0
    def __len__(self): return self.samples_per_epoch
    def __iter__(self):
        for _ in range(self.samples_per_epoch):
            if self.position==self.count:
                self.order=torch.randperm(self.count,generator=self.generator).tolist();self.position=0
            result=self.order[self.position];self.position+=1
            yield result

def epoch_pass(model,loader,optimizer=None,scaler=None):
    training=optimizer is not None;model.train(training);total=correct=top5=0;loss_sum=0.;started=time.monotonic();skipped=0
    with torch.set_grad_enabled(training):
        for batch_index,(x,n,y) in enumerate(loader,1):
            x=x.cuda();y=y.cuda()
            if training:x=augment_batch(x);optimizer.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                logits=model(x,n);loss=nn.functional.cross_entropy(logits,y)
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite loss')
            if training:skipped+=int(not backward_step(model,optimizer,loss,scaler))
            size=len(y);total+=size;loss_sum+=loss.item()*size;correct+=(logits.argmax(1)==y).sum().item();top5+=(logits.topk(5,dim=1).indices==y[:,None]).any(1).sum().item()
            if training and batch_index%50==0:print(f'  {"train" if training else "validation"} batch {batch_index}/{min(len(loader),150)}',flush=True)
            if training and batch_index>=150:break
    return {'samples':total,'loss':loss_sum/total,'top1':correct/total,'top5':top5/total,'seconds':time.monotonic()-started,'skipped_amp_updates':skipped}

def train_model(tag,layers,hidden):
    random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)
    model=DeepTrajectoryGRU(input_size=254,hidden_size=hidden,layers=layers,classes=700,dropout=.5).cuda();model.set_statistics(statistics)
    sampler=ContinuousShuffledSampler(len(train_data),samples_per_epoch=600,seed=42)
    loader=DataLoader(train_data,batch_size=4,sampler=sampler,shuffle=False,num_workers=0,collate_fn=FrameLimitCollator(augment=True))
    optimizer=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.001);scaler=torch.amp.GradScaler('cuda')
    directory=OUTPUT/tag;directory.mkdir();records=[];best=float('inf');stale=0;selected=0
    config={'architecture':'notebook-deep-gru-trajectory-v1','model':{'input_size':254,'hidden_size':hidden,'layers':layers,'classes':700,'dropout':.5},'preprocessing':{**manifest['identity']['preprocessing'],'max_frames':83,'sampling':'uniform ordered indices of cached sequence including endpoints; no upsampling'},'source_cache_max_frames':128,'training_sampling_order':'temporal augmentation on cached sequence, then 83-frame cap','parent_cache_id':manifest['cache_id'],'seed':42,'batch_size':4,'train_batches_per_epoch_cap':150,'sampler':'continuous seeded shuffle, carry position between epochs','epochs_cap':300,'loader_workers':0,'optimizer':'AdamW','learning_rate':.001,'weight_decay':.001,'temporal_augmentation':TEMPORAL_CONFIG,'regularization':PROFILES['regularized-v1'],'selection':'minimum validation cross entropy; patience 40; minimum 38 epochs; cap 300','geometry_features':list(FEATURES),'geometry_statistics':statistics}
    for epoch in range(1,301):
        tr=epoch_pass(model,loader,optimizer,scaler);val=epoch_pass(model,val_loader)
        row={'epoch':epoch,'train_augmented':tr,'val':val};records.append(row)
        (directory/'history.json').write_text(json.dumps(records,indent=2))
        state={'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},'config':config,'labels':labels,'epoch':epoch,'metrics':row}
        tmp=directory/'latest.tmp';torch.save(state,tmp);os.replace(tmp,directory/'latest.pt')
        if val['loss']<best-.001:
            best=val['loss'];stale=0;selected=epoch
            tmp=directory/'best.tmp';torch.save(state,tmp);os.replace(tmp,directory/'best-backbone.pt')
        else:stale+=1
        (directory/'status.json').write_text(json.dumps({'status':'training','epoch':epoch,'selected_epoch':selected},indent=2))
        print(f"{tag} epoch {epoch:02}: train {tr['top1']:.2%} val {val['top1']:.2%} val loss {val['loss']:.4f} duration {tr['seconds']+val['seconds']:.0f}s",flush=True)
        if epoch>=38 and stale>=40:break
    (directory/'status.json').write_text(json.dumps({'status':'backbone-complete','epoch':epoch,'selected_epoch':selected},indent=2))
    del model,optimizer,scaler;torch.cuda.empty_cache()
    return records
histories={}
for tag,layers,hidden in [('gru_2x256',2,256),('gru_5x512',5,512)]:
    histories[tag]=train_model(tag,layers,hidden)

## Fit frozen classifiers and evaluate

The readout is refit using original training clips and 23 geometry values, L2 logistic regression C=0.1. Final accuracy, top-5, loss, macro/weighted F1, precision, recall and per-class metrics are reported on official splits. The test set was inspected previously; results remain useful as a comparison, not as a new blind benchmark.

In [ ]:
def collect(model,loader):
    model.eval();h=[];g=[];ys=[]
    with torch.inference_mode():
        for x,n,y in loader:
            x=x.cuda()
            with torch.autocast('cuda',dtype=torch.float16): z=model.encode(x,n)
            h.append(z.float().cpu().numpy());g.append(trajectory_batch(x,n));ys.extend(y.tolist())
    return np.concatenate(h),np.concatenate(g),np.asarray(ys,dtype=np.int64)

def metrics(model,loader):
    h,g,y=collect(model,loader);z=model.standardized_geometry(torch.from_numpy(g).cuda()).detach().cpu().numpy()
    with torch.no_grad():logits=nn.functional.linear(torch.from_numpy(np.concatenate([h,z],axis=1)).float(),model.classifier.weight.cpu(),model.classifier.bias.cpu())
    p=logits.softmax(1);pred=p.argmax(1).numpy();precision,recall,f1,support=precision_recall_fscore_support(y,pred,labels=np.arange(700),zero_division=0)
    return {'samples':len(y),'correct':int((pred==y).sum()),'top1':float((pred==y).mean()),'top5':float((p.topk(5,1).indices.numpy()==y[:,None]).any(1).mean()),'loss':float(nn.functional.cross_entropy(logits,torch.from_numpy(y)).item()),'macro_precision':float(precision.mean()),'macro_recall':float(recall.mean()),'macro_f1':float(f1.mean()),'weighted_f1':float(np.average(f1,weights=support)), 'per_class':{s:{'support':int(support[i]),'precision':float(precision[i]),'recall':float(recall[i]),'f1':float(f1[i])} for i,s in enumerate(names)}}

test_index=json.loads((PARENT/'test-index.json').read_text());test_data=SelectedDataset('test',test_index)
test_people={test_index[i]['participant_id'] for i in test_data.rows};assert not test_people&(people['train']|people['val'])
test_loader=DataLoader(test_data,batch_size=16,shuffle=False,num_workers=0,collate_fn=FrameLimitCollator())
results={}
for tag,layers,hidden in [('gru_2x256',2,256),('gru_5x512',5,512)]:
    directory=OUTPUT/tag;state=torch.load(directory/'best-backbone.pt',map_location='cpu',weights_only=True)
    model=DeepTrajectoryGRU(**state['config']['model']).cuda();model.load_state_dict(state['model']);model.eval()
    h,g,y=collect(model,clean_loader);z=model.standardized_geometry(torch.from_numpy(g).cuda()).detach().cpu().numpy()
    with warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        head=LogisticRegression(C=.1,solver='lbfgs',max_iter=2000,random_state=42).fit(np.concatenate([h,z],1),y)
    assert np.array_equal(head.classes_,np.arange(700))
    with torch.no_grad():model.classifier.weight.copy_(torch.from_numpy(head.coef_).float().cuda());model.classifier.bias.copy_(torch.from_numpy(head.intercept_).float().cuda())
    model.gru.requires_grad_(False)
    final={'model':{k:v.detach().cpu() for k,v in model.state_dict().items()},'config':state['config'],'labels':labels,'epoch':state['epoch'],'readout':'frozen GRU plus L2 logistic regression C=0.1'}
    torch.save(final,directory/'model.pt')
    result={'epoch':state['epoch'],'parameters':sum(p.numel() for p in model.parameters()),'checkpoint_sha256':sha256(directory/'model.pt'),'metrics':{s:metrics(model,loader) for s,loader in [('train',clean_loader),('val',val_loader),('test',test_loader)]}}
    results[tag]=result;(directory/'report.json').write_text(json.dumps(result,indent=2));(directory/'status.json').write_text(json.dumps({'status':'complete','selected_epoch':state['epoch']},indent=2))
    for temporary in ('latest.pt','best-backbone.pt'):(directory/temporary).unlink(missing_ok=True)
    print(tag,[(s,round(x['top1']*100,2),round(x['macro_f1']*100,2),x['samples']) for s,x in result['metrics'].items()],flush=True)
(OUTPUT/'comparison.json').write_text(json.dumps({'status':'complete','models':results},indent=2))

## Accuracy, loss and final metrics

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4))
for tag,history in histories.items():
    x=[r['epoch'] for r in history]
    axes[0].plot(x,[r['val']['loss'] for r in history],label=tag+' validation');axes[0].plot(x,[r['train_augmented']['loss'] for r in history],linestyle='--',label=tag+' train augmented')
    axes[1].plot(x,[r['val']['top1'] for r in history],label=tag+' validation');axes[1].plot(x,[r['train_augmented']['top1'] for r in history],linestyle='--',label=tag+' train augmented')
for ax,title in zip(axes,['Cross entropy','Top-1 accuracy']):ax.set(title=title,xlabel='Epoch');ax.legend();ax.grid(alpha=.2)
plt.tight_layout();plt.show()
for tag,r in results.items():
    print(tag,'selected epoch',r['epoch'],'parameters',r['parameters'])
    for split,m in r['metrics'].items():print(split,{k:round(m[k],4) for k in ('top1','top5','loss','macro_precision','macro_recall','macro_f1','weighted_f1')},'n=',m['samples'])